In [1]:
print ('Hello World')

Hello World


In [ ]:
pip install torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118

In [7]:
!pip install torchvision



[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip



   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.7 MB ? eta -:--:--
   ------------ --------------------------- 0.5/1.7 MB 493.7 kB/s eta 0:00:03
   ------------ --------------------------- 0.5/1.7 MB 493.7 kB/s eta 0:00:03
   ------------------ --------------------- 0.8/1.7 MB 493.7 kB/s eta 0:00:02
   ------------------ --------------------- 0.8/1.7 MB 493.7 kB/s eta 0:00:02
   ------------------ --------------------- 0.8/1.7 MB 493.7 kB/s eta 0:00:02
   ------------------------ --------------- 1.0/1.7 MB 498.4 kB/s eta 0:00:02
   ------------------------ --------------- 1.0/1.7 MB 498.4 kB/s eta 0:00:02
   ------------------------------ -----

In [8]:
!pip install timm


[notice] A new release of pip is available: 25.0.1 -> 25.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached timm-1.0.15-py3-none-any.whl.metadata (52 kB)
  Using cached huggingface_hub-0.30.2-py3-none-any.whl.metadata (13 kB)
  Using cached safetensors-0.5.3-cp38-abi3-win_amd64.whl.metadata (3.9 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
Using cached timm-1.0.15-py3-none-any.whl (2.4 MB)
Using cached huggingface_hub-0.30.2-py3-none-any.whl (481 kB)
Using cached safetensors-0.5.3-cp38-abi3-win_amd64.whl (308 kB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import timm  # for EfficientNet and others
import os

In [ ]:
# Configuration
model_name = 'resnet50'  # Change to 'resnet50' 'densenet121', 'efficientnet_b0', 'inception_v3', 'vgg16'
num_classes = 3  # Change according to your dataset
batch_size = 32
epochs = 10
lr = 0.001

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# Datasets
data_dir = 'D:/Data_Analysis/DATASET '
image_datasets = {
    x: datasets.ImageFolder(os.path.join(data_dir, x), data_transforms[x])
    for x in ['train', 'val']
}
dataloaders = {
    x: DataLoader(image_datasets[x], batch_size=batch_size, shuffle=True)
    for x in ['train', 'val']
}

In [ ]:
# Transforms
data_transforms = {
    'train': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
    'val': transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406],
                             [0.229, 0.224, 0.225])
    ]),
}

In [ ]:
# Load Model
def get_model(name, num_classes):
    if name.startswith("efficientnet"):
        model = timm.create_model(name, pretrained=True)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    elif name.startswith("resnet"):
        model = getattr(models, name)(pretrained=True)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name.startswith("densenet"):
        model = getattr(models, name)(pretrained=True)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    elif name == "inception_v3":
        model = models.inception_v3(pretrained=True, aux_logits=False)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif name.startswith("vgg"):
        model = getattr(models, name)(pretrained=True)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    else:
        raise ValueError("Unsupported model type")
    return model

In [ ]:
model = get_model(model_name, num_classes).to(device)


In [ ]:
# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=lr)

# Training loop
for epoch in range(epochs):
    print(f"\nEpoch {epoch + 1}/{epochs}")
    for phase in ['train', 'val']:
        if phase == 'train':
            model.train()
        else:
            model.eval()

        running_loss = 0.0
        running_corrects = 0

        for inputs, labels in dataloaders[phase]:
            inputs = inputs.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            with torch.set_grad_enabled(phase == 'train'):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                _, preds = torch.max(outputs, 1)
                if phase == 'train':
                    loss.backward()
                    optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            running_corrects += torch.sum(preds == labels.data)

        epoch_loss = running_loss / len(image_datasets[phase])
        epoch_acc = running_corrects.double() / len(image_datasets[phase])
        print(f'{phase.capitalize()} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')